# Event displays for jets with unexpected flavour content

Standalone RDataFrame version of the IFN clustering from `jet-flv-test-histmaker.py`
(hadronisation partons or primary hadrons → flavoured pseudojets → exclusive 2-jet `ee_kt` + IFN).
Events where the jet flavour content does not match the expectation for the sample are drawn as a
2D event display in the plane spanned by the two jets:

* x axis: direction of jet 1, y axis: component of jet 2 orthogonal to jet 1.
* Partons/hadrons are arrows (projected momentum), coloured by the jet they are clustered into and labelled by name
  (g, u, b̄, ρ⁺, B̄⁰, ...; PDG IDs without a name are shown as numbers). Bold: quarks, and hadrons containing the Higgs
  decay flavour if that is s, c or b.
  Dashed arrows point mostly out of the plane, p_T > p_L (momentum transverse / longitudinal to the jet plane,
  i.e. out of / within it; also printed in the legend), so their drawn length is misleading.
* The events are sorted into categories by the jet flavour content. The two jets always have opposite content
  (flavour conservation), so for H → qq̄ the q content is ±n in the two jets:

  | category | q content | other flavours |
  |---|---|---|
  | `00_expected` | ±1 | none |
  | `01_q_plus_other` | ±1 | present (e.g. spectator quarks of hadrons, B⁻ = bū) |
  | `02_q_lost_neutral` | 0 | none (`g \| g`) |
  | `03_q_lost_other` | 0 | present |
  | `04_q_multiple` | ±2 or more (extra qq̄ pair) | any |

  H → gg (parton level only): `00_expected` (no net flavour) and `01_flavoured`.

Up to `N_PER_TYPE` events are drawn per category, round-robin over the flavour patterns in it (jet flavours, e.g.
`u bbar | ubar b`, most common first), so that different patterns are shown.
Plots go to `outputs/jet-test/event_displays/<level>_ifn/<sample>/<category>/` (one PDF per event plus one combined PDF
per category), with a `README.md` listing the categories and their most common patterns.

## Usage

Run it interactively, or in batch from this directory (after `source setup.sh`). The sample (default Hbb) and the
clustering input (`LEVEL`: `parton`, the default, or `hadron`) are chosen with environment variables; the other
settings (`N_FILES`, `N_EVENTS`, `N_PER_TYPE`) are in the config cell below. 20k events take about 4 minutes.

```bash
SAMPLE=wzp6_ee_nunuH_Hcc_ecm240 LEVEL=hadron jupyter nbconvert --to notebook --execute \
    --ExecutePreprocessor.timeout=1400 --output jet-event-display_Hcc_hadron.ipynb jet-event-display.ipynb
```

`--output` writes the executed notebook to a new file, so the checked-in notebook stays unchanged; use `--inplace` instead
to overwrite it. Several samples can run in parallel, each writes to its own plot directory.

In [1]:
import os
import glob
from collections import Counter

import numpy as np
import ROOT
from particle import Particle

ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)
# With a local build older than the nightly stack, loading libFCCAnalyses prints ~300 kB of
# "Info in <TCling::LoadPCM>" messages, which fill JupyROOT's output pipe and hang the kernel.
# A clean rebuild (fccanalysis build -c) gets rid of them; only show warnings and above anyway.
ROOT.gErrorIgnoreLevel = ROOT.kWarning
# Load libFastJet first, otherwise libFCCAnalyses resolves it to the central
# cvmfs copy, which lacks the FlavInfo symbols needed by the IFN clustering.
ROOT.gSystem.Load("libFastJet")
ROOT.gSystem.Load("libFCCAnalyses")
ROOT.dummyLoader()

True

In [2]:
SAMPLE = os.environ.get("SAMPLE", "wzp6_ee_nunuH_Hbb_ecm240")  # e.g. SAMPLE=... jupyter nbconvert --execute ...
LEVEL = os.environ.get("LEVEL", "parton")  # clustering input: "parton" (hadronisation partons) or "hadron" (primary hadrons)
INPUT_DIR = f"/eos/experiment/fcc/ee/generation/DelphesEvents/pre_summer2026_training/IDEA/{SAMPLE}"
N_FILES = 1        # number of input files to chain
N_EVENTS = 20000   # events to scan, -1 for all events in the chained files
N_PER_TYPE = 10    # max. number of events to draw per category

PLOT_DIR = os.path.join("..", "..", "..", "outputs", "jet-test", "event_displays", f"{LEVEL}_ifn", SAMPLE)
os.makedirs(PLOT_DIR, exist_ok=True)

FLAVOURS = ["u", "d", "s", "c", "b"]


HIGGS_FLAVOUR = SAMPLE.split("_H")[1][0]  # e.g. "b" for wzp6_ee_nunuH_Hbb_ecm240, "g" for Hgg


if LEVEL not in ("parton", "hadron"):
    raise ValueError(f"Unknown LEVEL {LEVEL!r}, use 'parton' or 'hadron'")
if LEVEL == "hadron" and HIGGS_FLAVOUR not in FLAVOURS:
    raise ValueError(f"H -> {HIGGS_FLAVOUR}{HIGGS_FLAVOUR} is not supported at hadron level")

# category (= directory name) -> description, in display order
if HIGGS_FLAVOUR in FLAVOURS:
    q = HIGGS_FLAVOUR
    CATEGORIES = {
        "00_expected": f"{q} content +-1, no other flavour (as expected)",
        "01_q_plus_other": f"{q} content +-1 plus other flavours",
        "02_q_lost_neutral": f"{q} content 0, no net flavour at all (g | g)",
        "03_q_lost_other": f"{q} content 0, other flavours present",
        "04_q_multiple": f"|{q} content| >= 2 (extra {q}{q}bar pair)",
    }
else:  # H -> gg
    CATEGORIES = {
        "00_expected": "no net flavour (as expected)",
        "01_flavoured": "net flavour in the jets",
    }


def categorise(contents):
    """Category of an event from its per-jet flavour contents ({flavour: [jet 1, jet 2]}).

    The two jets always have opposite contents (flavour conservation), so |q content| is the same for both.
    """
    other = any(c != 0 for f, cs in contents.items() if f != HIGGS_FLAVOUR for c in cs)
    if HIGGS_FLAVOUR not in FLAVOURS:
        return "01_flavoured" if other else "00_expected"
    n_q = max(abs(c) for c in contents[HIGGS_FLAVOUR])
    if n_q >= 2:
        return "04_q_multiple"
    if n_q == 1:
        return "01_q_plus_other" if other else "00_expected"
    return "03_q_lost_other" if other else "02_q_lost_neutral"


CATEGORIES

{'00_expected': 'b content +-1, no other flavour (as expected)',
 '01_q_plus_other': 'b content +-1 plus other flavours',
 '02_q_lost_neutral': 'b content 0, no net flavour at all (g | g)',
 '03_q_lost_other': 'b content 0, other flavours present',
 '04_q_multiple': '|b content| >= 2 (extra bbbar pair)'}

In [3]:
files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.root")))[:N_FILES]
chain = ROOT.TChain("events")
for f in files:
    chain.Add(f)

SELECTIONS = {
    "parton": "FCCAnalyses::MCParticle::get_hadronisation_partons(Particle, _Particle_parents.index)",
    "hadron": "FCCAnalyses::MCParticle::get_primary_hadrons(Particle, _Particle_parents.index)",
}

df = ROOT.RDataFrame(chain)
if N_EVENTS > 0:
    df = df.Range(N_EVENTS)  # Range needs single-threaded RDF, so no EnableImplicitMT here

df = (df.Define("entry", "rdfentry_")
        .Define("particles", SELECTIONS[LEVEL])
        .Define("particle_px", "FCCAnalyses::MCParticle::get_px(particles)")
        .Define("particle_py", "FCCAnalyses::MCParticle::get_py(particles)")
        .Define("particle_pz", "FCCAnalyses::MCParticle::get_pz(particles)")
        .Define("particle_m", "FCCAnalyses::MCParticle::get_mass(particles)")
        .Define("particle_pdg", "FCCAnalyses::MCParticle::get_pdg(particles)")
        .Define("pseudo_jets", "FCCAnalyses::JetClusteringUtils::set_pseudoJets_xyzm_flav(particle_px, particle_py, particle_pz, particle_m, particle_pdg)")
        .Define("clustered_jets", "JetClustering::flv_ifn_clustering_ee_kt(2, 2, 0, 0)(pseudo_jets)")
        .Define("jets", "FCCAnalyses::JetClusteringUtils::get_pseudoJets(clustered_jets)")
        .Define("jets_constituents", "FCCAnalyses::JetClusteringUtils::get_constituents(clustered_jets)")
        .Define("jets_flavour", "FCCAnalyses::JetClusteringUtils::get_flavours(jets)"))
for var in ["px", "py", "pz", "e"]:
    df = df.Define(f"jets_{var}", f"FCCAnalyses::JetClusteringUtils::get_{var}(jets)")
for f in FLAVOURS:
    df = df.Define(f"jets_{f}_content", f"FCCAnalyses::JetClusteringUtils::get_{f}_content(jets)")

n_total = df.Count()
columns = (["entry", "particle_px", "particle_py", "particle_pz", "particle_pdg", "jets_constituents", "jets_flavour"]
           + [f"jets_{v}" for v in ["px", "py", "pz", "e"]] + [f"jets_{f}_content" for f in FLAVOURS])
events = df.Filter("jets.size() == 2", "two jets").AsNumpy(columns)  # triggers the event loop
n_events = len(events["entry"])
print(f"{SAMPLE}, {LEVEL} level: {n_events} of {n_total.GetValue()} events with two jets")

#--------------------------------------------------------------------------
#                         FastJet release 3.5.1
#                 M. Cacciari, G.P. Salam and G. Soyez                  
#     A software package for jet finding and analysis at colliders      
#                           https://fastjet.fr                           
#	                                                                      
# Please cite EPJC72(2012)1896 [arXiv:1111.6097] if you use this package
# for scientific work and optionally PLB641(2006)57 [hep-ph/0512210].   
#                                                                       
# FastJet is provided without warranty under the GNU GPL v2 or higher.  
# It uses T. Chan's closest pair algorithm, S. Fortune's Voronoi code
# and 3rd party plugin jet algorithms. See COPYING file for details.
#--------------------------------------------------------------------------


wzp6_ee_nunuH_Hbb_ecm240, parton level: 20000 of 20000 events with two jets


In [4]:
# Sort the events into categories, and within a category group them by flavour pattern (jet flavours as strings).
def flavour_pattern(i_ev):
    return " | ".join(sorted(str(s).strip() for s in events["jets_flavour"][i_ev]))


categories = {name: {} for name in CATEGORIES}  # category -> pattern -> event indices
for i_ev in range(n_events):
    contents = {f: [int(c) for c in events[f"jets_{f}_content"][i_ev]] for f in FLAVOURS}
    categories[categorise(contents)].setdefault(flavour_pattern(i_ev), []).append(i_ev)
categories = {name: dict(sorted(patterns.items(), key=lambda p: -len(p[1]))) for name, patterns in categories.items()}
counts = {name: sum(len(indices) for indices in patterns.values()) for name, patterns in categories.items()}

for name, description in CATEGORIES.items():
    print(f"{name:18s} {counts[name]:6d} ({counts[name] / n_events:6.1%})  {description}")
    for pattern, indices in list(categories[name].items())[:5]:
        print(f"{'':18s} {len(indices):6d}           {pattern}")

00_expected         19120 ( 95.6%)  b content +-1, no other flavour (as expected)
                    19120           [b ] | [bbar ]
01_q_plus_other        12 (  0.1%)  b content +-1 plus other flavours
                        3           [s b ] | [sbar bbar ]
                        3           [u bbar ] | [ubar b ]
                        3           [s bbar ] | [sbar b ]
                        2           [d bbar ] | [dbar b ]
                        1           [c b ] | [cbar bbar ]
02_q_lost_neutral     863 (  4.3%)  b content 0, no net flavour at all (g | g)
                      863           [g] | [g]
03_q_lost_other         3 (  0.0%)  b content 0, other flavours present
                        2           [s ] | [sbar ]
                        1           [d ] | [dbar ]
04_q_multiple           2 (  0.0%)  |b content| >= 2 (extra bbbar pair)
                        2           [b b ] | [bbar bbar ]


In [5]:
import re

JET_COLOURS = [ROOT.kRed + 1, ROOT.kAzure + 2]
PARTON_NAMES = {1: "d", 2: "u", 3: "s", 4: "c", 5: "b", 6: "t", 21: "g"}
QUARK_NAMES = {"d": "down", "u": "up", "s": "strange", "c": "charm", "b": "bottom"}


def particle_label(pdg):
    """TLatex label and whether to highlight it (quarks; hadrons containing the Higgs decay flavour if
    that is s, c or b, since nearly every light hadron contains a u or d).

    Partons: "g", "b", "#bar{b}"; hadrons: their LaTeX name from the particle package without the
    mass, e.g. "#rho^{+}", "#bar{B}^{0}"; the PDG ID for anything unknown.
    """
    name = PARTON_NAMES.get(abs(pdg))
    if name is not None:
        return (f"#bar{{{name}}}" if pdg < 0 else name), abs(pdg) <= 6
    try:
        p = Particle.from_pdgid(pdg)
    except Exception:
        return str(pdg), False
    latex = re.sub(r"\(\d+\)", "", p.latex_name)  # drop the mass, e.g. rho(770)
    latex = latex.replace("\\overline", "#bar").replace("\\", "#")
    latex = latex.replace("^#prime", "'")  # e.g. eta', TLatex needs braces otherwise
    has_q = HIGGS_FLAVOUR in ("s", "c", "b") and getattr(p.pdgid, f"has_{QUARK_NAMES[HIGGS_FLAVOUR]}")
    return latex, has_q


def event_location(entry):
    """Input file and entry within it, e.g. for edm4hep2json."""
    chain.LoadTree(int(entry))
    return chain.GetFile().GetName(), int(entry) - chain.GetChainOffset()


def jet_plane(j1, j2):
    """Orthonormal basis (e1, e2, n) with e1 along jet 1 and jet 2 in the (e1, e2) plane."""
    e1 = j1 / np.linalg.norm(j1)
    v = j2 - j2.dot(e1) * e1
    if np.linalg.norm(v) < 1e-6 * np.linalg.norm(j2):  # back-to-back jets: plane not defined, pick any
        v = np.cross(e1, [0., 0., 1.] if abs(e1[2]) < 0.9 else [1., 0., 0.])
    e2 = v / np.linalg.norm(v)
    return e1, e2, np.cross(e1, e2)


def content_string(data, i_ev, i_jet):
    return " ".join(f"{f}{int(data[f'jets_{f}_content'][i_ev][i_jet]):+d}" for f in FLAVOURS)


def event_display(data, i_ev):
    """Event display of event i_ev in data (columns from AsNumpy)."""
    entry = int(data["entry"][i_ev])
    jets = [np.array([data[f"jets_{v}"][i_ev][i] for v in ["px", "py", "pz"]]) for i in range(2)]
    particles = np.array([np.asarray(data[f"particle_{v}"][i_ev]) for v in ["px", "py", "pz"]]).T
    pdgs = np.asarray(data["particle_pdg"][i_ev]).astype(int)
    jet_of = np.full(len(pdgs), -1)
    for i_jet, constituents in enumerate(data["jets_constituents"][i_ev]):
        jet_of[np.asarray(constituents)] = i_jet

    e1, e2, n = jet_plane(*jets)
    proj = lambda p: (p @ e1, p @ e2)
    px, py = proj(particles)
    out = np.abs(particles @ n)
    jx, jy = zip(*(proj(j) for j in jets))

    r = 1.15 * max(np.max(np.abs(np.concatenate([px, py, jx, jy]))), 1.)
    c = ROOT.TCanvas(f"evt_{SAMPLE}_{entry}", "", 800, 800)
    c.SetMargin(0.12, 0.05, 0.1, 0.1)
    frame = c.DrawFrame(-r, -r, r, r)
    frame.SetTitle(f"{SAMPLE}, {LEVEL} level, entry {entry};p #upoint #hat{{e}}_{{1}} (jet 1) [GeV];p #upoint #hat{{e}}_{{2}} [GeV]")
    keep = [frame]

    for i_jet in range(2):  # jet axes: thick, translucent
        a = ROOT.TArrow(0., 0., jx[i_jet], jy[i_jet], 0.025, "|>")
        a.SetLineWidth(8)
        a.SetLineColorAlpha(JET_COLOURS[i_jet], 0.3)
        a.SetFillColorAlpha(JET_COLOURS[i_jet], 0.3)
        a.Draw()
        keep.append(a)

    for x, y, o, pdg, i_jet in zip(px, py, out, pdgs, jet_of):
        colour = JET_COLOURS[i_jet] if i_jet >= 0 else ROOT.kGray + 2
        a = ROOT.TArrow(0., 0., x, y, 0.012, "|>")
        a.SetLineWidth(2)
        a.SetLineColor(colour)
        a.SetFillColor(colour)
        if o > np.hypot(x, y):
            a.SetLineStyle(2)
        a.Draw()
        text, highlight = particle_label(pdg)
        label = ROOT.TLatex(x * 1.04, y * 1.04, text)
        label.SetTextAlign((31 if x < 0 else 11) + (0 if y < 0 else 2) - 1)  # place label away from the origin
        label.SetTextSize(0.032 if highlight else 0.022)
        label.SetTextFont(62 if highlight else 42)
        label.SetTextColor(colour)
        label.Draw()
        keep += [a, label]

    leg = ROOT.TLegend(0.14, 0.12, 0.6, 0.27)
    leg.SetBorderSize(0)
    leg.SetFillStyle(0)
    leg.SetTextSize(0.025)
    for i_jet in range(2):
        line = ROOT.TLine()
        line.SetLineColor(JET_COLOURS[i_jet])
        line.SetLineWidth(3)
        leg.AddEntry(line, f"jet {i_jet + 1}: {content_string(data, i_ev, i_jet)}", "l")
        keep.append(line)
    dashed = ROOT.TLine()
    dashed.SetLineStyle(2)
    # condition used for the dashed arrows above, spelled out so that a single plot is self-explanatory
    leg.AddEntry(dashed, "p_{T} > p_{L}  (w.r.t. jet plane)", "l")
    leg.Draw()
    keep += [leg, dashed]

    c._keep = keep  # keep the primitives alive
    return c

In [6]:
import re
import shutil


# Remove the category directories of a previous run (also the old per-type ones).
for d in os.listdir(PLOT_DIR):
    if re.match(r"\d{2}_", d) and os.path.isdir(os.path.join(PLOT_DIR, d)):
        shutil.rmtree(os.path.join(PLOT_DIR, d))


def pick_events(patterns, n):
    """Up to n event indices, round-robin over the patterns (most common first), so that different patterns are shown."""
    queues = [list(indices) for indices in patterns.values()]
    picked = []
    while len(picked) < n and any(queues):
        for queue in queues:
            if queue and len(picked) < n:
                picked.append(queue.pop(0))
    return picked


def draw_group(data, indices, dirname):
    """Draw the events into PLOT_DIR/dirname, plus a combined PDF."""
    if not indices:
        return []
    group_dir = os.path.join(PLOT_DIR, dirname)
    os.makedirs(group_dir, exist_ok=True)
    group_canvases = []
    for i_ev in indices:
        entry = int(data["entry"][i_ev])
        fname, local_entry = event_location(entry)
        print(f"entry {entry} ({flavour_pattern(i_ev)})  ->  edm4hep2json -e {local_entry} -l Particle -o out.json {fname}")
        c = event_display(data, i_ev)
        c.SaveAs(os.path.join(group_dir, f"entry_{entry}.pdf"))
        c.Draw()
        group_canvases.append(c)
    combined = os.path.join(group_dir, f"all_events_{dirname}.pdf")
    group_canvases[0].Print(combined + "[")
    for c in group_canvases:
        c.Print(combined, f"Title:{c.GetName()}")
    group_canvases[-1].Print(combined + "]")
    return group_canvases


def short(pattern):
    return pattern.replace("[", "").replace(" ]", "").replace("]", "")


readme = [f"{LEVEL.capitalize()}-level IFN jets in {SAMPLE} ({n_events} events), one subdirectory per category of the "
          f"jet flavour content. Each holds up to {N_PER_TYPE} event displays (`entry_<n>.pdf`), taken from as many "
          f"different flavour patterns as possible, and all of them combined (`all_events_*.pdf`). "
          f"Generated by `examples/FCCee/flav_aware_jets/jet-event-display.ipynb`.", ""]

canvases = []
for name, description in CATEGORIES.items():
    n = counts[name]
    picked = pick_events(categories[name], N_PER_TYPE)
    top = ", ".join(f"{short(p)} ({len(indices)})" for p, indices in list(categories[name].items())[:3])
    readme.append(f"- `{name}/`: {description}; {n} event{'s' if n != 1 else ''} ({n / n_events:.1%}), "
                  f"{len(picked)} drawn" + (f"; most common: {top}" if top else ""))
    print(f"--- {name}: drawing {len(picked)} of {n} events")
    canvases += draw_group(events, picked, name)

with open(os.path.join(PLOT_DIR, "README.md"), "w") as f:
    f.write("\n".join(readme) + "\n")

--- 00_expected: drawing 10 of 19120 events
entry 0 ([b ] | [bbar ])  ->  edm4hep2json -e 0 -l Particle -o out.json root://eospublic.cern.ch//eos/experiment/fcc/ee/generation/DelphesEvents/pre_summer2026_training/IDEA/wzp6_ee_nunuH_Hbb_ecm240/events_505359172.root
entry 1 ([b ] | [bbar ])  ->  edm4hep2json -e 1 -l Particle -o out.json root://eospublic.cern.ch//eos/experiment/fcc/ee/generation/DelphesEvents/pre_summer2026_training/IDEA/wzp6_ee_nunuH_Hbb_ecm240/events_505359172.root
entry 2 ([b ] | [bbar ])  ->  edm4hep2json -e 2 -l Particle -o out.json root://eospublic.cern.ch//eos/experiment/fcc/ee/generation/DelphesEvents/pre_summer2026_training/IDEA/wzp6_ee_nunuH_Hbb_ecm240/events_505359172.root
entry 3 ([b ] | [bbar ])  ->  edm4hep2json -e 3 -l Particle -o out.json root://eospublic.cern.ch//eos/experiment/fcc/ee/generation/DelphesEvents/pre_summer2026_training/IDEA/wzp6_ee_nunuH_Hbb_ecm240/events_505359172.root
entry 4 ([b ] | [bbar ])  ->  edm4hep2json -e 4 -l Particle -o out.json 

--- 01_q_plus_other: drawing 10 of 12 events
entry 1235 ([s b ] | [sbar bbar ])  ->  edm4hep2json -e 1235 -l Particle -o out.json root://eospublic.cern.ch//eos/experiment/fcc/ee/generation/DelphesEvents/pre_summer2026_training/IDEA/wzp6_ee_nunuH_Hbb_ecm240/events_505359172.root
entry 1923 ([u bbar ] | [ubar b ])  ->  edm4hep2json -e 1923 -l Particle -o out.json root://eospublic.cern.ch//eos/experiment/fcc/ee/generation/DelphesEvents/pre_summer2026_training/IDEA/wzp6_ee_nunuH_Hbb_ecm240/events_505359172.root
entry 13273 ([s bbar ] | [sbar b ])  ->  edm4hep2json -e 13273 -l Particle -o out.json root://eospublic.cern.ch//eos/experiment/fcc/ee/generation/DelphesEvents/pre_summer2026_training/IDEA/wzp6_ee_nunuH_Hbb_ecm240/events_505359172.root
entry 471 ([d bbar ] | [dbar b ])  ->  edm4hep2json -e 471 -l Particle -o out.json root://eospublic.cern.ch//eos/experiment/fcc/ee/generation/DelphesEvents/pre_summer2026_training/IDEA/wzp6_ee_nunuH_Hbb_ecm240/events_505359172.root
entry 17000 ([c b ]